# 01. Agent Identity & DIDs

Every entity in Arc has a cryptographic identity. Not a username. Not an API key. A **Decentralized Identifier (DID)** backed by an Ed25519 keypair, generated locally, and stable across the lifetime of the agent. This is the *first* of the four pillars (Identity, Sign, Authorize, Audit) and the foundation everything else stands on.

Arc's identity primitives live in `arctrust` — the leaf shared library every other Arc package depends on. arcagent uses them to bind agents to keys, arcrun uses them to label events, arcteam uses them to scope channels. If `arctrust.identity` is broken, nothing else works.

**You will learn:**
- The Arc DID format: `did:arc:{org}:{type}/{hash}` and why each segment matters
- How to construct an `AgentIdentity` from scratch with `AgentIdentity.generate`
- How `parse_did` and `validate_did` differ and when to reach for each
- How `ChildIdentity` derives short-lived ephemeral identities for spawned subagents
- How to persist and load identities from disk with secure permissions (0600/0700)
- Where identity sits in the four-pillar story (sign / authorize / audit)

Everything in this notebook runs locally. No API key. No network. Just real cryptography.

## 1. Setup

The setup cell below is the standard Arc walkthrough preamble: it locates the repo root, adds every package's `src/` (and `tests/` where present) to `sys.path`, and loads any `.env` file at the root. This is identical across every notebook so you can skim past it once you've seen it.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Quick smoke test — confirm `arctrust` imports cleanly and the symbols we need are present.

In [2]:
import arctrust
from arctrust import (
    AgentIdentity,
    ChildIdentity,
    KeyPair,
    generate_did,
    parse_did,
    validate_did,
    derive_child_identity,
)

print(f"arctrust version: {arctrust.__version__}")
print(
    "Loaded:",
    [
        AgentIdentity.__name__,
        ChildIdentity.__name__,
        KeyPair.__name__,
        generate_did.__name__,
        parse_did.__name__,
        validate_did.__name__,
        derive_child_identity.__name__,
    ],
)

arctrust version: 0.9.0
Loaded: ['AgentIdentity', 'ChildIdentity', 'KeyPair', 'generate_did', 'parse_did', 'validate_did', 'derive_child_identity']


## 2. What is a DID?

A **DID** (Decentralized Identifier) is a self-sovereign identity string. Unlike an email address or a UUID minted by some central registry, a DID is derived directly from cryptographic material the holder controls. If you have the private key, you *are* the DID. There's no provider to revoke you, no central server to go down.

Arc uses a custom DID method, `did:arc:`, with this shape:

```
did:arc:{org}:{type}/{hash}
```

| Segment | Meaning | Example |
|---|---|---|
| `did:arc:` | Method prefix — every Arc DID starts here | `did:arc:` |
| `{org}` | Deployment organization | `default`, `doe`, `acme` |
| `{type}` | Agent role / classification | `executor`, `planner`, `child` |
| `{hash}` | First 8 hex chars of SHA-256(public_key) | `9b43ee77` |

The hash suffix is **deterministic** — the same Ed25519 public key always produces the same DID. That gives you a stable identifier without a registry. The first three segments are *labels* (you choose them); the hash is *content-addressed* (the math chooses it).

Why DIDs and not just UUIDs?

1. **Verifiable.** A DID is bound to a keypair — you can sign with it.
2. **Federation-friendly.** Arc deployments can interoperate without a shared registry.
3. **Compliance-aligned.** Aligns with W3C DID Core and NIST 800-53 IA-2/IA-5.
4. **Audit-clean.** Every audit event is keyed on a DID; same identity → same key, forever.

## 3. Generating identities

There are two ways to get a DID:

1. **Top-down**: call `AgentIdentity.generate(org, agent_type)` — this generates a    fresh Ed25519 keypair, derives the DID, and gives you a fully-functional identity    that can sign and verify.
2. **Bottom-up**: build the keypair separately and call `generate_did(verify_key,    org=..., agent_type=...)` to derive just the DID string.

The top-down path is what 99% of code uses. The bottom-up path matters when you already have a key (e.g. loaded from a vault) and just need the matching DID string.

In [3]:
# Top-down: one call gives you a working identity.
alice = AgentIdentity.generate(org="acme", agent_type="executor")
print("DID:        ", alice.did)
print("Public key: ", alice.public_key.hex()[:32], "...")
print("Can sign?   ", alice.can_sign)

DID:         did:arc:acme:executor/dfdf6da2
Public key:  6d058780c465d210fb241e08bcbfb231 ...
Can sign?    True


Three things to notice in the output:

- The DID matches the format `did:arc:acme:executor/{8-hex}`.
- The public key is exactly 32 bytes (Ed25519 standard). We slice it for readability.
- `can_sign` is `True` because `generate()` produces a *full* identity with a private key.   Verify-only identities (constructed from a public key alone) have `can_sign = False`.

Now the bottom-up path. We'll create a `KeyPair` directly, then derive the DID. This is what `from_config` does internally when loading a key from a vault — the key comes first, the DID label is computed afterward.

In [4]:
from nacl.signing import SigningKey
from arctrust import generate_keypair

# Generate a fresh keypair the cryptographic way.
kp: KeyPair = generate_keypair()
print(f"Public key length:  {len(kp.public_key)} bytes")
print(f"Private key length: {len(kp.private_key)} bytes")

# Now derive a DID from that key.
verify_key = SigningKey(kp.private_key).verify_key
did = generate_did(verify_key, org="acme", agent_type="planner")
print("Derived DID:", did)

Public key length:  32 bytes
Private key length: 32 bytes
Derived DID: did:arc:acme:planner/1952169a


**Determinism check.** The DID hash is `sha256(public_key)[:8]`. Same key in, same DID out. This is the property that lets us treat the DID as a content-addressed handle.

In [5]:
did_again = generate_did(verify_key, org="acme", agent_type="planner")
assert did == did_again, "DIDs must be deterministic for the same key"
print("Deterministic:", did == did_again)

# Different keys -> different DIDs.
other_kp = generate_keypair()
other_did = generate_did(
    SigningKey(other_kp.private_key).verify_key, org="acme", agent_type="planner"
)
assert other_did != did
print("Different key, different DID:", other_did)

Deterministic: True
Different key, different DID: did:arc:acme:planner/6c1057dd


## 4. Parsing and validating DIDs

Two functions, one for each job:

- `parse_did(did) -> dict[str, str]` — split a known-good DID into `{org, agent_type, hash}`.
  Raises `ValueError` on anything malformed. Use this when you already trust the input   and want to access its parts.
- `validate_did(did) -> str` — check that a DID is well-formed and return it. **Empty   string is valid**, signaling auto-generation. Use this at config-load time to gate   whether to load a key or generate a new one.

Treat them as a pair: `validate_did` for sanitizing untrusted input, `parse_did` for extracting structure once you've validated.

In [6]:
# Parse a well-formed DID.
parts = parse_did("did:arc:acme:planner/9b43ee77")
print(parts)
assert parts == {"org": "acme", "agent_type": "planner", "hash": "9b43ee77"}

{'org': 'acme', 'agent_type': 'planner', 'hash': '9b43ee77'}


Round-trip a real identity through `parse_did` to confirm the segments match the values we passed into `generate()`.

In [7]:
alice_parts = parse_did(alice.did)
print("alice DID parts:", alice_parts)
assert alice_parts["org"] == "acme"
assert alice_parts["agent_type"] == "executor"
assert len(alice_parts["hash"]) == 8
int(alice_parts["hash"], 16)  # hash must be valid hex
print("All segments check out.")

alice DID parts: {'org': 'acme', 'agent_type': 'executor', 'hash': 'dfdf6da2'}
All segments check out.


### `validate_did` — accepts empty for auto-generate

Arc's config story is: a user can leave `did = ""` in their TOML, and the agent will generate a new identity at first boot, then write the DID back into the config. `validate_did` encodes that contract — empty is *valid*, anything non-empty must be a well-formed `did:arc:` string.

In [8]:
# Empty string is valid (signals auto-generate).
assert validate_did("") == ""

# Well-formed DID round-trips unchanged.
good = "did:arc:acme:planner/9b43ee77"
assert validate_did(good) == good

print("Empty validated:", repr(validate_did("")))
print("Good validated:", validate_did(good))

Empty validated: ''
Good validated: did:arc:acme:planner/9b43ee77


### Error cases — what malformed input looks like

The next three cells each raise `ValueError`. They're labelled `# this raises` so the reader knows what to expect. We catch the exception and print the message — that lets the notebook execute end-to-end without crashing while still showing real failure modes.

In [9]:
# this raises — wrong DID method (not did:arc:)
try:
    validate_did("did:web:example.com")
except ValueError as e:
    print("ValueError:", e)

ValueError: Invalid DID format: 'did:web:example.com'. Must be 'did:arc:{org}:{type}/{hash}' or empty for auto-generation.


In [10]:
# this raises — missing the /{hash} segment
try:
    validate_did("did:arc:org:executor")
except ValueError as e:
    print("ValueError:", e)

ValueError: Malformed DID structure: 'did:arc:org:executor'. Expected 'did:arc:{org}:{type}/{hash}'.


In [11]:
# this raises — parse_did rejects the same input
try:
    parse_did("did:arc:only_three")
except ValueError as e:
    print("ValueError:", e)

ValueError: Malformed DID structure: 'did:arc:only_three'. Expected 'did:arc:{org}:{type}/{hash}'.


Notice how `parse_did` and `validate_did` share the same malformed-DID detection but differ on **what counts as input**: `validate_did("")` is fine; `parse_did("")` would not be (an empty string has no parts to extract). Pick the right tool for the job.

## 5. Signing and verifying with `AgentIdentity`

An `AgentIdentity` is more than a DID label — it's a *capability*. The full version carries a private signing key, so it can produce signatures. The verify-only version carries just the public key, so it can check signatures but not produce them. Both are the same class; the difference is whether `_signing_key` is set.

Signatures are 64 bytes of Ed25519 — deterministic, no per-signature randomness needed.

In [12]:
msg = b"alice authorizes tool_call_42"
sig = alice.sign(msg)

print(f"Signature is {len(sig)} bytes")
print(f"Signature hex: {sig.hex()[:48]}...")

# Verify with the same identity that signed.
assert alice.verify(msg, sig) is True
print("Self-verify: OK")

Signature is 64 bytes
Signature hex: dc80f729aa3f7962b3e8fffeba0414bce03632b86b0b2e25...
Self-verify: OK


Cross-check: a verify-only twin of Alice (no private key, just the public one) can verify her signatures but not produce them. This is the pattern you use when an agent has someone else's public key — you can audit their signatures, but you can't impersonate them.

In [13]:
# Build a verify-only twin from alice's public key.
alice_verify_only = AgentIdentity(
    did=alice.did,
    public_key=alice.public_key,
    _signing_key=None,
)

print("can_sign?", alice_verify_only.can_sign)
assert alice_verify_only.verify(msg, sig) is True
print("Verify-only verified the signature: OK")

can_sign? False
Verify-only verified the signature: OK


In [14]:
# this raises — verify-only identity cannot sign
try:
    alice_verify_only.sign(b"forged")
except ValueError as e:
    print("ValueError:", e)

ValueError: Cannot sign: no private key available (verify-only identity)


**Tampering check.** Flip a single bit in the message and verification fails. Ed25519 is doing its job — the signature is bound to the exact bytes that were signed.

In [15]:
tampered = b"alice authorizes tool_call_43"  # 42 -> 43
result = alice.verify(tampered, sig)
print("Verify of tampered message:", result)
assert result is False

Verify of tampered message: False


## 6. Parent-child hierarchies with `ChildIdentity`

Long-lived agents spawn short-lived helpers. A planner spawns an executor for one task. A team coordinator spawns a worker for a single batch. These ephemeral workers need their *own* identity — they should sign their own audit events and be revocable without touching the parent — but minting a fresh top-level keypair every time would be wasteful and would make audit lineage hard to trace.

Arc's answer is `ChildIdentity` + `derive_child_identity`. Given:

- the parent's secret key (`parent_sk_bytes`)
- a per-spawn nonce (`spawn_id`)
- a wallclock TTL (`wallclock_timeout_s`)

we deterministically derive a 32-byte Ed25519 seed using HKDF-SHA256, then build a child DID of the form `did:arc:delegate:child/{hex8}`. The derivation is:

```
PRK   = HMAC-SHA256(salt="arc.spawn", IKM=parent_sk)
T(1)  = HMAC-SHA256(PRK, info=spawn_id || 0x01)   # 32 bytes -> child seed
```

**Security properties (ASI-03 / ASI-07):**

- The child seed is unpredictable to anyone without the parent's secret key.
- The child seed is deterministic for the same `(parent_sk, spawn_id)` — useful for   reproducibility and for *re-deriving* an identity rather than storing it.
- Different `spawn_id` values yield uncorrelated child seeds (the nonce is the only   thing that should change between spawns).

In [16]:
# Pretend alice is the parent and we want to spawn a child for task #1.
assert alice._signing_key is not None
alice_secret = alice._signing_key.encode()

child = derive_child_identity(
    parent_sk_bytes=alice_secret,
    spawn_id="task-001",
    wallclock_timeout_s=300,
)
print("Child DID:    ", child.did)
print("Child SK len: ", len(child.sk_bytes))
print("Child TTL:    ", child.ttl_s, "seconds")

Child DID:     did:arc:delegate:child/e5eb767b
Child SK len:  32
Child TTL:     300 seconds


Determinism: same `(parent_sk, spawn_id)` always produces the same child. This is the property that lets the parent re-derive a child identity without storing it.

In [17]:
child_again = derive_child_identity(
    parent_sk_bytes=alice_secret,
    spawn_id="task-001",
    wallclock_timeout_s=300,
)
assert child.did == child_again.did
assert child.sk_bytes == child_again.sk_bytes
print("Re-derived identical child:", child.did == child_again.did)

# Different spawn_id -> different child.
child2 = derive_child_identity(
    parent_sk_bytes=alice_secret,
    spawn_id="task-002",
    wallclock_timeout_s=300,
)
assert child2.did != child.did
assert child2.sk_bytes != child.sk_bytes
print("Different spawn_id yields different child:", child2.did)

Re-derived identical child: True
Different spawn_id yields different child: did:arc:delegate:child/99ffadab


`ChildIdentity` is a Pydantic model, so you can serialize it with `model_dump()` for passing between processes (e.g. handing the child seed to a freshly-forked subprocess) or for logging. **Never log the seed in production** — but for local development and debugging it's safe to inspect.

In [18]:
child_dict = child.model_dump()
# Mask the secret for display
redacted = {**child_dict, "sk_bytes": f"<{len(child_dict['sk_bytes'])} bytes redacted>"}
print(redacted)

# Round-trip through model_validate.
rebuilt = ChildIdentity.model_validate(child_dict)
assert rebuilt.did == child.did
assert rebuilt.sk_bytes == child.sk_bytes
print("Round-trip via Pydantic: OK")

{'did': 'did:arc:delegate:child/e5eb767b', 'sk_bytes': '<32 bytes redacted>', 'ttl_s': 300, 'clearance': <Classification.UNCLASSIFIED: 0>}
Round-trip via Pydantic: OK


Promote the child to a full `AgentIdentity` so it can sign and be verified. The seed is 32 bytes, which is exactly what `SigningKey` and `KeyPair.from_seed` expect.

In [19]:
child_signing = SigningKey(child.sk_bytes)
child_identity = AgentIdentity(
    did=child.did,
    public_key=bytes(child_signing.verify_key),
    _signing_key=child_signing,
)
child_msg = b"child authorizes tool_call_99"
child_sig = child_identity.sign(child_msg)
assert child_identity.verify(child_msg, child_sig)
print("Child identity signed and verified its own message:", child_identity.did)

# And alice (the parent) can verify it too — by re-deriving and checking the public key.
expected_child_pk = bytes(SigningKey(child.sk_bytes).verify_key)
assert child_identity.public_key == expected_child_pk
print("Parent re-derivation matches child public key.")

Child identity signed and verified its own message: did:arc:delegate:child/e5eb767b
Parent re-derivation matches child public key.


## 7. Identity persistence

Identities are useless if they don't survive a restart. `AgentIdentity.save_keys` and `AgentIdentity.load_keys` are the contract for filesystem persistence:

- The key directory is created at **0700** (owner-only access).
- The `.key` file (private signing seed) is written at **0600** (owner read/write).
- The `.pub` file (public key) is written at **0644** (publicly readable).
- On load, **0600** is enforced — anything looser raises `ValueError("insecure permissions")`.

These permissions matter: NIST 800-53 IA-5 requires that authentication material be protected at rest, and the SecurityModule layer expects this on warm start. We'll use a temporary directory so this notebook leaves nothing on disk.

In [20]:
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp(prefix="arc_identity_demo_"))
print("Demo key dir:", tmp)

# Save alice's keypair into the temp dir.
alice.save_keys(tmp)
saved = sorted(p.name for p in tmp.iterdir())
print("Files written:", saved)

Demo key dir: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arc_identity_demo_h80sku0l
Files written: ['did_arc_acme_executor_dfdf6da2.key', 'did_arc_acme_executor_dfdf6da2.pub']


Verify the file permissions match the contract.

In [21]:
import stat

safe_name = alice.did.replace(":", "_").replace("/", "_")
key_file = tmp / f"{safe_name}.key"
pub_file = tmp / f"{safe_name}.pub"

key_mode = stat.S_IMODE(key_file.stat().st_mode)
pub_mode = stat.S_IMODE(pub_file.stat().st_mode)
dir_mode = stat.S_IMODE(tmp.stat().st_mode)

print(f"Dir  mode: {oct(dir_mode)}  (expect 0o700)")
print(f"Key  mode: {oct(key_mode)}  (expect 0o600)")
print(f"Pub  mode: {oct(pub_mode)}  (expect 0o644)")

assert dir_mode == 0o700
assert key_mode == 0o600
assert pub_mode == 0o644

Dir  mode: 0o700  (expect 0o700)
Key  mode: 0o600  (expect 0o600)
Pub  mode: 0o644  (expect 0o644)


Load the identity back from disk. `load_keys` reads the file, reconstructs the `SigningKey`, and returns a fully-functional `AgentIdentity`.

In [22]:
alice_reloaded = AgentIdentity.load_keys(alice.did, tmp)
print("Reloaded DID:    ", alice_reloaded.did)
print("Public key match:", alice_reloaded.public_key == alice.public_key)
print("Can sign?       :", alice_reloaded.can_sign)

# Round-trip a signature through the reloaded identity.
rt_msg = b"after restart"
rt_sig = alice_reloaded.sign(rt_msg)
assert alice.verify(rt_msg, rt_sig), (
    "Reloaded identity should produce signatures the original verifies"
)
print("Reloaded identity signed a message and the original verified it.")

Reloaded DID:     did:arc:acme:executor/dfdf6da2
Public key match: True
Can sign?       : True
Reloaded identity signed a message and the original verified it.


**Insecure permissions are rejected.** Loosen the key file to 0644 and `load_keys` refuses — preventing a class of attack where a misconfigured deployment leaks signing keys to other local users.

In [23]:
key_file.chmod(0o644)
try:
    AgentIdentity.load_keys(alice.did, tmp)
except ValueError as e:
    print("ValueError:", e)
finally:
    key_file.chmod(0o600)  # restore for cleanup

ValueError: Key file has insecure permissions: 0o644. Expected 0o600 (owner read/write only).


### `from_config` — the production wiring

In production, agents don't call `generate()` and `save_keys()` directly. They go through `AgentIdentity.from_config(...)`, which:

1. Validates `config.did` (empty = auto-generate).
2. If a vault resolver is provided and `config.vault_path` is set, loads from the vault.
3. Otherwise loads from `config.key_dir` on disk.
4. If no DID was set, generates a fresh one, saves keys, and writes the new DID back    into the TOML config file (so subsequent boots load the same identity).

We'll demonstrate the auto-generate path with a synthetic config object — this is the first-boot experience for a fresh agent.

In [24]:
class DemoConfig:
    """Stand-in for arcagent.IdentityConfig — anything with did/key_dir/vault_path works."""

    def __init__(self, did: str, key_dir: str, vault_path: str = "") -> None:
        self.did = did
        self.key_dir = key_dir
        self.vault_path = vault_path


config_dir = Path(tempfile.mkdtemp(prefix="arc_config_demo_"))
config_file = config_dir / "arcagent.toml"
config_file.write_text(
    f'[identity]\ndid = ""\nkey_dir = "{config_dir / "keys"}"\n',
    encoding="utf-8",
)

demo_cfg = DemoConfig(did="", key_dir=str(config_dir / "keys"))
fresh = AgentIdentity.from_config(
    demo_cfg,
    org="acme",
    agent_type="executor",
    config_path=config_file,
)
print("Generated fresh identity:", fresh.did)
print()
print("Updated config file content:")
print(config_file.read_text(encoding="utf-8"))

Generated fresh identity: did:arc:acme:executor/3db02392

Updated config file content:
[identity]
did = "did:arc:acme:executor/3db02392"
key_dir = "/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arc_config_demo_d88t1y0z/keys"



Notice that the empty `did = ""` was rewritten with the generated DID — the next process boot will *load* this identity instead of generating a new one. This is how agents get a stable identity across restarts without operator intervention.

## 8. Identity in the four pillars

Identity is the first pillar. The other three (Sign, Authorize, Audit) all key off it:

| Pillar | How it uses identity |
|---|---|
| **Identity** | Every entity has an `AgentIdentity` with a unique DID. (this notebook) |
| **Sign** | Loaded artifacts (skills, manifests) are verified against the issuer's DID public key. See **02-keypairs-signing**. |
| **Authorize** | `PolicyPipeline` evaluates a `ToolCall(agent_did=...)` — DID is the principal. See **03-policy-pipeline**. |
| **Audit** | Every `AuditEvent` carries `actor_did`. Sinks chain events keyed on identity. See **04-audit-sinks**. |

Concretely, here's a sketch of what each pillar consumes from this notebook:

In [25]:
# Sketch — illustrative only. Each of these is covered in depth in its own notebook.

# Sign: verify a signed payload using the identity's public key.
from arctrust import sign as raw_sign, verify as raw_verify

assert alice._signing_key is not None
payload = b"skill-bundle-v1.0.3-manifest"
payload_sig = raw_sign(payload, alice._signing_key.encode())
assert raw_verify(payload, payload_sig, alice.public_key)
print("[Sign]      manifest verified for", alice.did)

# Authorize: a ToolCall carries caller_did; the policy pipeline reads it.
from arctrust import ToolCall

tool_call = ToolCall(
    tool_name="http.get",
    arguments={"url": "https://example.com"},
    agent_did=alice.did,
    session_id="sess-001",
    classification="UNCLASSIFIED",
)
print("[Authorize] policy will evaluate agent_did =", tool_call.agent_did)

# Audit: an AuditEvent records the actor's DID.
from arctrust import AuditEvent

evt = AuditEvent(
    actor_did=alice.did,
    action="tool.call",
    target=tool_call.tool_name,
    outcome="allow",
)
print("[Audit]     event recorded actor_did =", evt.actor_did)

[Sign]      manifest verified for did:arc:acme:executor/dfdf6da2
[Authorize] policy will evaluate agent_did = did:arc:acme:executor/dfdf6da2
[Audit]     event recorded actor_did = did:arc:acme:executor/dfdf6da2


Three lines, three pillars, all pointing back at the same DID. That's the design: identity is the *primary key* for the whole security story. Lose track of who an actor is and the rest of the system has nothing to anchor on.

**Tier note (ADR-019).** Personal, Enterprise, and Federal tiers all run this same code path. The difference is *stringency* — Federal requires FIPS-validated crypto, Enterprise enables the manifest-issuer pubkey check, Personal allows self-signed bundles with an audit warning. But every tier still verifies, authorizes, audits, and identifies through `arctrust`.

### Cleanup

Wipe the temp directories so this notebook is leave-no-trace.

In [26]:
import shutil

for d in (tmp, config_dir):
    if d.exists():
        shutil.rmtree(d)
        print("Removed:", d)

Removed: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arc_identity_demo_h80sku0l
Removed: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arc_config_demo_d88t1y0z


## 9. Summary

**What you saw in this notebook:**

- The Arc DID format `did:arc:{org}:{type}/{hash}` and why each segment matters.
- `AgentIdentity.generate(...)` — the one-call path to a working identity with sign/verify.
- `generate_did(verify_key, ...)` — the bottom-up path when you already have a key.
- `parse_did` (for trusted input) and `validate_did` (for config-time sanitization,   with empty meaning auto-generate).
- Sign / verify / tamper-detection roundtrips through an `AgentIdentity`.
- Verify-only identities (no private key) for auditing other entities' signatures.
- `ChildIdentity` and `derive_child_identity` for short-lived spawned subagents,   derived deterministically via HKDF-SHA256 with `(parent_sk, spawn_id, ttl)`.
- `save_keys` / `load_keys` / `from_config` — persistence with 0600/0700 enforcement   and DID-back-to-config rewriting on first boot.
- Where identity sits in the four-pillar story: it's the primary key for sign,   authorize, and audit.

**Public API surface covered:**

- `AgentIdentity` (constructor, `generate`, `sign`, `verify`, `save_keys`, `load_keys`,   `from_config`, `can_sign`)
- `ChildIdentity` (Pydantic model, `model_dump`, `model_validate`)
- `KeyPair` (constructor via `generate_keypair`)
- `generate_did`
- `parse_did`
- `validate_did`
- `derive_child_identity` (positional + keyword forms)
- Brief touch on `sign`, `verify`, `ToolCall`, `AuditEvent` (deep-dives in later notebooks)

**Next:** `02-keypairs-signing.ipynb` goes deeper on Ed25519 itself — `KeyPair`, raw `sign` / `verify`, the `load_operator_pubkey` and `load_issuer_pubkey` trust-store primitives, and how signing fits into the artifact-loading pipeline.